# 原始现金预测到 EMA/HOLD 提案：好坏区间对照

只读 Qlib Recorder 来源。比较同一事前趋势与年份下的原始现金、接受现金以及两者差，不推断更换处理链的账户收益。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R

root=Path.cwd();card=root/"research/stage1_f696_smoothing_interval_attribution_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());dep=plan["dependencies"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and all(v==0 for v in plan["budget"].values())
bridge=root/dep["bridge_card"];interval=root/dep["interval_card"]
assert sha(bridge/"plan.json")==dep["bridge_plan_sha256"] and sha(bridge/"result.json")==dep["bridge_result_sha256"]
assert sha(interval/"plan.json")==dep["interval_plan_sha256"] and sha(interval/"result.json")==dep["interval_result_sha256"]
bp=json.loads((bridge/"plan.json").read_text());br=json.loads((bridge/"result.json").read_text())
ip=json.loads((interval/"plan.json").read_text());ir=json.loads((interval/"result.json").read_text())
assert br["identity_gate_pass"] and br["sessions"]==944 and ir["rolling126_windows"]==819
src=bp["sources"];account_src=ip["source"]
assert all(k in src for k in ["action_card","action_experiment_id","action_recorder_id","action_artifacts","f696_card","f696_experiment_id","f696_recorder_id","f696_frame_sha256"])
assert all(k in account_src for k in ["card","summary_sha256","experiment_id","recorder_id","daily_sha256","nas_sha256","provider_manifest_sha256"])
action=root/src["action_card"];f696=root/src["f696_card"];account=root/account_src["card"]
assert sha(action/"summary.json")==src["action_summary_sha256"]
assert sha(f696/"summary.json")==src["f696_summary_sha256"]
assert sha(account/"summary.json")==account_src["summary_sha256"]
sa=json.loads((action/"summary.json").read_text());sf=json.loads((f696/"summary.json").read_text());sc=json.loads((account/"summary.json").read_text())
assert sa["nas_evidence"]["sha256"]==src["action_archive_sha256"] and sa["nas_evidence"]["size_bytes"]==src["action_archive_size_bytes"]
assert sf["nas_evidence"]["sha256"]==src["f696_archive_sha256"] and sf["nas_evidence"]["size_bytes"]==src["f696_archive_size_bytes"]
assert sc["nas_readback"]["verified"] and sc["nas_readback"]["sha256"]==account_src["nas_sha256"]
assert sha(root/src["provider"]/"qlib_store_manifest.json")==src["provider_manifest_sha256"]==account_src["provider_manifest_sha256"]
ar=action/"qlib_recorder"/src["action_experiment_id"]/src["action_recorder_id"]/"artifacts"
fr=f696/"qlib_recorder"/src["f696_experiment_id"]/src["f696_recorder_id"]/"artifacts"
cr=account/"qlib_recorder"/account_src["experiment_id"]/account_src["recorder_id"]/"artifacts"
for name,digest in src["action_artifacts"].items():assert sha(ar/name)==digest,name
assert sha(fr/"inputs/frame")==src["f696_frame_sha256"]
assert sha(cr/"analysis/daily")==account_src["daily_sha256"]
assert sha(cr/"analysis/rolling126")==dep["account_rolling_sha256"]
assert sha(cr/"analysis/periods")==dep["account_periods_sha256"]
qlib.init(provider_uri=str((root/src["provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None)
with R.uri_context(str(action/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["action_experiment_id"],recorder_id=src["action_recorder_id"])
 market=rec.load_object("inputs/market")
 raw=rec.load_object("inputs/raw_predictions")["frozen_current"]
 accepted=rec.load_object("inputs/source_targets")["frozen_current"]
with R.uri_context(str(f696/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["f696_experiment_id"],recorder_id=src["f696_recorder_id"])
 frame=rec.load_object("inputs/frame")
with R.uri_context(str(account/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=account_src["experiment_id"],recorder_id=account_src["recorder_id"])
 daily=rec.load_object("analysis/daily").sort_values("trade_date").reset_index(drop=True)
 rolling=rec.load_object("analysis/rolling126")
 periods=rec.load_object("analysis/periods")
dates=pd.DatetimeIndex(market.signal_date)
assert len(dates)==len(frame)==len(daily)==944 and len(rolling)==819
assert dates.equals(pd.DatetimeIndex(frame.signal_date)) and dates.equals(pd.DatetimeIndex(daily.signal_date))
assert len(periods.loc[periods.kind.eq("quarter")])==16
assert periods.loc[periods.kind.eq("year") & periods.partial_boundary_period,"period"].astype(str).tolist()==["2023","2026"]
columns=["soxl_weight","soxx_weight","cash_weight","tail_probability","alpha_probability"]
raw=raw.reindex(dates).loc[:,columns];accepted=accepted.reindex(dates).loc[:,columns]
frame_values=frame.set_index(dates).loc[:,columns]
identity={m:float((accepted[m]-frame_values[m]).abs().max()) for m in columns}
assert max(identity.values())==0
x=pd.DataFrame({"raw_cash":raw.cash_weight.to_numpy(float),"accepted_cash":accepted.cash_weight.to_numpy(float),
                "accepted_minus_raw_cash":(accepted.cash_weight-raw.cash_weight).to_numpy(float)},index=dates)
assert x.notna().all().all()
calendar=pd.DatetimeIndex(D.calendar(start_time="2011-01-03",end_time="2026-10-09",freq="day"))
close=D.features(["SOXX"],["$close"],start_time="2011-01-03",end_time="2026-10-09",freq="day")["$close"].droplevel("instrument").sort_index()
assert close.index.equals(calendar) and close.notna().all()
annual=[]
for year in (2023,2024,2025,2026):
 y=x.loc[x.index.year==year]
 annual.append({"year":year,"sessions":len(y),"partial_endpoint":year in (2023,2026),
                **{m:float(y[m].mean()) for m in x.columns}})
windows=[]
for i in range(819):
 rw=rolling.iloc[i];segment=daily.iloc[i:i+126]
 assert pd.Timestamp(segment.trade_date.iloc[0])==pd.Timestamp(rw.first_trade)
 assert abs(float(segment.paired_log_excess.sum())-float(rw.relative_log_excess))<1e-10
 signal=pd.Timestamp(segment.signal_date.iloc[0]);pos=calendar.get_loc(signal);assert pos>=20
 r5=float(close.iloc[pos]/close.iloc[pos-5]-1);r20=float(close.iloc[pos]/close.iloc[pos-20]-1)
 state="uptrend" if r5>=0 and r20>=0 else "rebound" if r5>=0 else "pullback" if r20>=0 else "decline"
 values=x.iloc[i:i+126]
 row={"year":int(pd.Timestamp(rw.first_trade).year),"start_state":state,"outcome":"weak" if rw.relative_log_excess<0 else "good" if rw.relative_log_excess>0 else "tie"}
 row.update({m:float(values[m].mean()) for m in x.columns})
 windows.append(row)
w=pd.DataFrame(windows)
assert w.outcome.value_counts().to_dict()=={"weak":564,"good":255}
cells=[]
for (state,year),g in w.groupby(["start_state","year"],sort=True):
 a=g[g.outcome.eq("weak")];b=g[g.outcome.eq("good")]
 c={"start_state":state,"year":int(year),"weak_windows":len(a),"good_windows":len(b),"matched":bool(len(a) and len(b))}
 if c["matched"]:c["weak_minus_good"]={m:float(a[m].mean()-b[m].mean()) for m in x.columns}
 cells.append(c)
matched=[c for c in cells if c["matched"]]
assert len(matched)==12 and all(c["good_windows"]==0 for c in cells if c["year"]==2023)
weights=np.array([min(c["weak_windows"],c["good_windows"]) for c in matched],dtype=float)
pooled={m:float(np.average([c["weak_minus_good"][m] for c in matched],weights=weights)) for m in x.columns}
signs={m:{"positive_cells":sum(c["weak_minus_good"][m]>0 for c in matched),"negative_cells":sum(c["weak_minus_good"][m]<0 for c in matched)} for m in x.columns}
assert abs(pooled["raw_cash"]+pooled["accepted_minus_raw_cash"]-pooled["accepted_cash"])<1e-12
result={"status":"read_only_smoothing_interval_cash_gap_attribution_no_policy",
        "plan_sha256":sha(plan_path),"source_target_f696_max_abs":identity,
        "sessions":944,"rolling126_windows":819,"weak_windows":564,"good_windows":255,
        "annual":annual,"matched_state_year_cells":len(matched),"matched_weak_minus_good":pooled,"matched_signs":signs,"cells":cells,
        "new_market_requests":0,"new_model_fits":0,"new_native_accounts":0,"new_bootstraps":0,"new_recorders":0,
        "policy_inference":False,"independent_validation":False,
        "limits":["The raw-to-accepted contrast is a proposal-coordinate difference, not a funded-account intervention",
                  "126-session windows overlap heavily and future outcomes define weak/good groups",
                  "2023 contains no good window in any state-year cell; no within-year matched effect there"]}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"matched":pooled,"signs":signs,"annual":annual})


2026/10/24 16:30:12 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[58308:MainThread](2026-10-24 16:30:12,408) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[58308:MainThread](2026-10-24 16:30:12,410) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[58308:MainThread](2026-10-24 16:30:12,410) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{'matched': {'raw_cash': 0.02549326033717679, 'accepted_cash': 0.027536160940432933, 'accepted_minus_raw_cash': 0.0020429006032561606}, 'signs': {'raw_cash': {'positive_cells': 9, 'negative_cells': 3}, 'accepted_cash': {'positive_cells': 9, 'negative_cells': 3}, 'accepted_minus_raw_cash': {'positive_cells': 9, 'negative_cells': 3}}, 'annual': [{'year': 2023, 'sessions': 250, 'partial_endpoint': True, 'raw_cash': 0.3720442542238977, 'accepted_cash': 0.3727750350982734, 'accepted_minus_raw_cash': 0.0007307808743756859}, {'year': 2024, 'sessions': 252, 'partial_endpoint': False, 'raw_cash': 0.5021799924734336, 'accepted_cash': 0.49939516130998995, 'accepted_minus_raw_cash': -0.0027848311634436572}, {'year': 2025, 'sessions': 250, 'partial_endpoint': False, 'raw_cash': 0.41008627790376406, 'accepted_cash': 0.41341247651248697, 'accepted_minus_raw_cash': 0.0033261986087229447}, {'year': 2026, 'sessions': 192, 'partial_endpoint': True, 'raw_cash': 0.32787481287286063, 'accepted_cash': 0.3244

## 中文结论

只读已验证的 `raw_predictions`、EMA/HOLD 后 `source_targets`、精确 f696 提案与原0bp Tiered账户。944日五输出的来源目标对 f696 提案最大差均为0；保留16季度、两端不完整年度及819个连续126日窗口。按窗口起点前已知的 SOXX 5/20 日走势符号及首交易年配对，12个格子同时有好坏窗口，2023四格没有好窗口。0新行情、拟合或账户。

按每格较小组窗口数加权，弱窗口减好窗口的**平均现金权重**：裸模型原始预测 **+0.02549**，接受的 f696 提案 **+0.02754**，EMA/HOLD 对两者差额的追加对比 **+0.00204**。三项在12格中均为9格正、3格负；年度接受减原始均值则2023/2024/2025/2026分别为+0.00073/-0.00278/+0.00333/-0.00345。来源平滑不是坏窗口现金提案差的主要形成位置；差异在原始配置头已大部分存在。

这只是冻结路径的权重对照，不是取消 EMA/HOLD 后的账户反事实。区间按未来收益标记且高度重叠，不能从+0.00204推断调平滑会获得多少净收益。固定版本决定 `descriptive_raw_allocator_cash_gap_dominates_postmodel_smoothing_gap_no_policy`：下一个真正因子机制应针对原配置头何时错误地偏向现金，并与自身风险容量一起验证；不能把平滑跨度或HOLD阈值在复用历史上调参当作alpha。需要新信息门和其自身完整0bp含费账户，SOXX目标未达成。
